In [42]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

In [43]:
# I tried a few different multilayer models, and they all underperformed compared to this one.
class SingleLayer(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, 8),
            nn.ReLU(),
            #nn.Dropout(0.4), # works slightly better without the dropout layer, and considering we're already doing mini-batching and a train-test split, I'm not too concerned about overfitting
            nn.Linear(8, 1))
    def forward(self, x):
        return(self.sequential(x))

In [44]:
mystery_data = pd.read_csv("FP_Data.csv")

mystery_data_onehot = pd.get_dummies(mystery_data)

y = mystery_data_onehot.pop("y")
X = mystery_data_onehot

X = normalize(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.25, random_state=28)

In [45]:
# How well does a linear model do, in terms of MSE, when trained on the train data?

from sklearn.linear_model import LinearRegression

baseline_lm = LinearRegression().fit(X_train, y_train)
lm_mse = np.mean((baseline_lm.predict(X_test) - y_test)**2)
print(f"Baseline Linear Model Validation MSE: {lm_mse:.4f}")

Baseline Linear Model Validation MSE: 125.3744


In [46]:
model = SingleLayer(11)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1) # This is, as far as I seen, the most widely used optimizer, though it is not what is used in the textbook
loss_fn = nn.MSELoss()

epochs = 50 # The model appears to stabilize by the 50th epoch
batch_size = 32

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.float32)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.float32)

for epoch in range(epochs):
    model.train()

    permutation = torch.randperm(X_train.size(0))
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        X_batch, y_batch = X_train[indices], y_train[indices]

        optimizer.zero_grad()
        output = model(X_batch).squeeze()
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test).squeeze()
        val_loss = loss_fn(val_output, y_test)
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {val_loss.item():.4f}")

Epoch 1/50 | Val Loss: 3144.7375
Epoch 2/50 | Val Loss: 3042.9150
Epoch 3/50 | Val Loss: 2900.8145
Epoch 4/50 | Val Loss: 2717.8787
Epoch 5/50 | Val Loss: 2493.9299
Epoch 6/50 | Val Loss: 2230.5071
Epoch 7/50 | Val Loss: 1934.0537
Epoch 8/50 | Val Loss: 1616.3492
Epoch 9/50 | Val Loss: 1288.9597
Epoch 10/50 | Val Loss: 970.4795
Epoch 11/50 | Val Loss: 689.0606
Epoch 12/50 | Val Loss: 472.3555
Epoch 13/50 | Val Loss: 341.1409
Epoch 14/50 | Val Loss: 291.5215
Epoch 15/50 | Val Loss: 290.4550
Epoch 16/50 | Val Loss: 293.1715
Epoch 17/50 | Val Loss: 275.6279
Epoch 18/50 | Val Loss: 243.3864
Epoch 19/50 | Val Loss: 206.1266
Epoch 20/50 | Val Loss: 180.9829
Epoch 21/50 | Val Loss: 173.8474
Epoch 22/50 | Val Loss: 179.8235
Epoch 23/50 | Val Loss: 191.0468
Epoch 24/50 | Val Loss: 201.5014
Epoch 25/50 | Val Loss: 204.3789
Epoch 26/50 | Val Loss: 200.5866
Epoch 27/50 | Val Loss: 190.9923
Epoch 28/50 | Val Loss: 177.3193
Epoch 29/50 | Val Loss: 164.1910
Epoch 30/50 | Val Loss: 152.5218
Epoch 31/5

In [47]:
# This is also just the last validation loss

model.eval()
with torch.no_grad():
    y_pred = model(X_test).squeeze()
    mse = loss_fn(y_pred, y_test)
    print(f"Test MSE: {mse.item():.4f}")

Test MSE: 130.1271


While the nueral network underperforms the baseline linear model run on the full dataset, if we look at the baseline linear model run only on the train data, then tested on the test data, they are comparable.

## 10-Fold Cross Validation

In [48]:
df = pd.read_csv("10-fold-cv.csv")

torch.manual_seed(28)
np.random.seed(28)

df_onehot = pd.get_dummies(df.drop(columns=["fold"]))
y = df_onehot.pop("y")
X = normalize(df_onehot)

k = 10
epochs = 50
batch_size = 32
nn_cv_results = []

In [49]:
for fold in range(1, k + 1):

    train_mask = (df["fold"] != fold).values
    test_mask = (df["fold"] == fold).values

    X_train = torch.tensor(X[train_mask], dtype=torch.float32)
    X_test = torch.tensor(X[test_mask], dtype=torch.float32)
    y_train = torch.tensor(np.asarray(y[train_mask]), dtype=torch.float32)
    y_test = torch.tensor(np.asarray(y[test_mask]), dtype=torch.float32)

    model = SingleLayer(11)
    optimizer = torch.optim.Adam(model.parameters(), lr = 0.1)
    loss_fn = nn.MSELoss()


    for epoch in range(epochs):
        model.train()

        permutation = torch.randperm(X_train.size(0))
        for i in range(0, X_train.size(0), batch_size):
            indices = permutation[i:i+batch_size]
            X_batch, y_batch = X_train[indices], y_train[indices]

            optimizer.zero_grad()
            output = model(X_batch).squeeze()
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test).squeeze()
        rmse = torch.sqrt(loss_fn(val_output, y_test)).item()
        mae = torch.mean(torch.abs(val_output - y_test)).item()

    nn_cv_results.append({"fold": fold, "model": "NeuralNetwork", "rmse": rmse, "mae": mae})

nn_cv_df = pd.DataFrame(nn_cv_results)
nn_cv_df.to_csv("reg_nn_cv_results.csv", index=False)
print(nn_cv_df)

   fold          model       rmse        mae
0     1  NeuralNetwork  13.407055  10.491920
1     2  NeuralNetwork   8.596800   6.787371
2     3  NeuralNetwork  10.652687   8.983821
3     4  NeuralNetwork  11.999849   9.914467
4     5  NeuralNetwork   9.882111   8.181055
5     6  NeuralNetwork  13.187973  10.798590
6     7  NeuralNetwork  11.688983   8.949135
7     8  NeuralNetwork  11.746499  10.263121
8     9  NeuralNetwork  12.276154   8.515909
9    10  NeuralNetwork   9.995507   8.458280
